## AI4Climate ML tutorial - template
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-03-16
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

Look at how the machine learning pipeline is different when we are working with gridded data.

Intro


### Prerequisites 
what background information is needed to go through the notebook


### Learning outcomes from completing the notebook

## Tutorial 
a balance of explanation and activity



#### Import libraries
Key libraries for this tutorial include:
- Xarray for loading the input dataset
- scikit learn for preparing the data for training
- pytorch for creating and training the neural network
- matplotlib and cartopy for visualising the results
- scores for evaluating model performance

In [24]:
import pathlib
import os
import datetime
import json
import re

In [3]:
import numpy 
import xarray

In [4]:
import matplotlib
import matplotlib.pyplot
import cartopy.crs

In [5]:
import sklearn
import sklearn.preprocessing
import sklearn.tree


In [6]:
import mlflow

In [7]:
import torch

The tutorial config from the JSON file.

In [8]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'jasmin',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/ssde/j25a/mmh_storage/ai4c_data/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer

In [9]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'weatherbench'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'weatherbench'
    return root_path

In [10]:
current_platform = tutorial_config['platform']

In [11]:
current_platform

'jasmin'

In [12]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/gws/ssde/j25a/mmh_storage/ai4c_data/weatherbench')

Define the key parameters for this experiment.

In [20]:
resolution_dict = {5.625: '5.625deg'}

In [113]:
weatherbench_dir = root_data_dir / resolution_dict[5.625]
print(weatherbench_dir.is_dir())
weatherbench_dir

True


PosixPath('/gws/ssde/j25a/mmh_storage/ai4c_data/weatherbench/5.625deg')

In [ ]:
var_list = {
    'temperature': [850, 500]
    'geopotential': [500],
}

In [36]:
pattern = re.compile(r"temperature" + r"_(\d{4})_5\.625deg\.nc")
pattern

re.compile(r'temperaturegs_(\d{4})_5\.625deg\.nc', re.UNICODE)

In [39]:
current_var = 'temperature'

In [40]:
pattern = re.compile(current_var + r"_(\d{4})_5\.625deg\.nc")
files_to_load = [ f1 for f1 in (root_data_dir / resolution_dict[5.625] / current_var).iterdir() if pattern.match(f1.name)]
temp_ds = xarray.open_mfdataset(files_to_load)
temp_ds

<xarray.Dataset> Size: 37GB
Dimensions:  (time: 350640, level: 13, lat: 32, lon: 64)
Coordinates:
  * time     (time) datetime64[ns] 3MB 1979-01-01 ... 2018-12-31T23:00:00
  * level    (level) int32 52B 50 100 150 200 250 300 ... 600 700 850 925 1000
  * lat      (lat) float64 256B -87.19 -81.56 -75.94 ... 75.94 81.56 87.19
  * lon      (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4
Data variables:
    t        (time, level, lat, lon) float32 37GB dask.array<chunksize=(8760, 13, 32, 64), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2020-03-03 19:31:21 GMT by grib_to_netcdf-2.16.0: /opt/ecmw...

In [50]:
temp_ds.loc[{'level': [500, 850] }]

<xarray.Dataset> Size: 6GB
Dimensions:  (time: 350640, level: 2, lat: 32, lon: 64)
Coordinates:
  * time     (time) datetime64[ns] 3MB 1979-01-01 ... 2018-12-31T23:00:00
  * level    (level) int32 8B 500 850
  * lat      (lat) float64 256B -87.19 -81.56 -75.94 ... 75.94 81.56 87.19
  * lon      (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4
Data variables:
    t        (time, level, lat, lon) float32 6GB dask.array<chunksize=(8760, 2, 32, 64), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2020-03-03 19:31:21 GMT by grib_to_netcdf-2.16.0: /opt/ecmw...

In [84]:
ds1 = temp_ds.loc[{'level': [850], 'time': slice(datetime.datetime(1980,1,1,0,0), datetime.datetime(1980,1,6,0,0))}]

In [85]:
ds1

<xarray.Dataset> Size: 993kB
Dimensions:  (time: 121, level: 1, lat: 32, lon: 64)
Coordinates:
  * time     (time) datetime64[ns] 968B 1980-01-01 ... 1980-01-06
  * level    (level) int32 4B 850
  * lat      (lat) float64 256B -87.19 -81.56 -75.94 ... 75.94 81.56 87.19
  * lon      (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4
Data variables:
    t        (time, level, lat, lon) float32 991kB dask.array<chunksize=(121, 1, 32, 64), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2020-03-03 19:31:21 GMT by grib_to_netcdf-2.16.0: /opt/ecmw...

In [77]:
(float(ds1['t'].mean().to_numpy()),
float(ds1['t'].std().to_numpy()))

(273.6348571777344, 14.708879470825195)

In [122]:
list(ds1['time'].values)

[np.datetime64('1980-01-01T00:00:00.000000000'),
 np.datetime64('1980-01-01T01:00:00.000000000'),
 np.datetime64('1980-01-01T02:00:00.000000000'),
 np.datetime64('1980-01-01T03:00:00.000000000'),
 np.datetime64('1980-01-01T04:00:00.000000000'),
 np.datetime64('1980-01-01T05:00:00.000000000'),
 np.datetime64('1980-01-01T06:00:00.000000000'),
 np.datetime64('1980-01-01T07:00:00.000000000'),
 np.datetime64('1980-01-01T08:00:00.000000000'),
 np.datetime64('1980-01-01T09:00:00.000000000'),
 np.datetime64('1980-01-01T10:00:00.000000000'),
 np.datetime64('1980-01-01T11:00:00.000000000'),
 np.datetime64('1980-01-01T12:00:00.000000000'),
 np.datetime64('1980-01-01T13:00:00.000000000'),
 np.datetime64('1980-01-01T14:00:00.000000000'),
 np.datetime64('1980-01-01T15:00:00.000000000'),
 np.datetime64('1980-01-01T16:00:00.000000000'),
 np.datetime64('1980-01-01T17:00:00.000000000'),
 np.datetime64('1980-01-01T18:00:00.000000000'),
 np.datetime64('1980-01-01T19:00:00.000000000'),
 np.datetime64('1980

In [100]:
ds1.loc[{'time': ds1['time'][12].values}]

<xarray.Dataset> Size: 9kB
Dimensions:  (level: 1, lat: 32, lon: 64)
Coordinates:
  * level    (level) int32 4B 850
  * lat      (lat) float64 256B -87.19 -81.56 -75.94 ... 75.94 81.56 87.19
  * lon      (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4
    time     datetime64[ns] 8B 1980-01-01T12:00:00
Data variables:
    t        (level, lat, lon) float32 8kB dask.array<chunksize=(1, 32, 64), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2020-03-03 19:31:21 GMT by grib_to_netcdf-2.16.0: /opt/ecmw...

In [80]:
((ds1['t'] - 273.634) / 14.7089).mean().to_numpy()

array(5.668598e-05, dtype=float32)

In [144]:
ds1['t'].loc[{'time': '1980-1-1'}]

<xarray.DataArray 't' (time: 24, level: 1, lat: 32, lon: 64)> Size: 197kB
dask.array<getitem, shape=(24, 1, 32, 64), dtype=float32, chunksize=(24, 1, 32, 64), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 192B 1980-01-01 ... 1980-01-01T23:00:00
  * level    (level) int32 4B 850
  * lat      (lat) float64 256B -87.19 -81.56 -75.94 ... 75.94 81.56 87.19
  * lon      (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4
Attributes:
    units:          K
    long_name:      Temperature
    standard_name:  air_temperature

In [92]:
list(ds1.data_vars)

['t']

In [93]:
era5_rename_lut = {
    'z': 'geopotential',
    't': 'temperature',
}

In [96]:
def get_rename_dict(xr_ds, var_lut):
    vars_present = list(xr_ds.data_vars)
    rename_lut = {k1: v1 for k1,v1 in var_lut.items() if k1 in vars_present}
    return rename_lut
                  

In [97]:
get_rename_dict(ds1, era5_rename_lut)

{'t': 'temperature'}

In [145]:
class WeatherbenchDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, time_period, var_dict, resolution_str, is_train=True):
        self._is_train=is_train

        self._stats_dict = {}
        self._ds_dict = {}

        for current_var, pl_list in var_dict.items():
            for current_pl in pl_list:
                pattern = re.compile(current_var + r"_(\d{4})_5\.625deg\.nc")
                files_to_load = [ f1 for f1 in (data_dir / current_var).iterdir() if pattern.match(f1.name)]
                temp_ds = xarray.open_mfdataset(files_to_load)
                temp_ds = temp_ds.rename(get_rename_dict(temp_ds, era5_rename_lut))
                subset_dict = {
                    'time': slice(time_period[0], time_period[1]),
                    'level': current_pl,
                }
                
                wb_ds = temp_ds.loc[subset_dict]
                var_str = f'{current_var}_{current_pl}'
                var_id = (current_var, current_pl)
                print(var_str)
                self._ds_dict[var_id] = wb_ds

                if is_train:
                    var_stats = (    
                        float(wb_ds[current_var].mean().to_numpy()),
                        float(wb_ds[current_var].std().to_numpy()),
                    )
                    
                    self._stats_dict[var_id] = {'mean': var_stats[0], 'std': var_stats[1]}
                    
        self._time_list = (self._ds_dict[ list(self._ds_dict.keys())[0] ]['time'].values)
        # iterate through dict
        # open dataset for variable
        # subset by level (if multilevel)
        # subset by time 
        #  store pointer to xarray dataset(s) in class for referencing metadata later 
        # if train, calculate mean and std dev, and save to stats dict, with var/level as key
                
        # concat into a numpy array
        

    def __str__(self):
        return str(self._wb_ds)

    def __repr_html__(self):
        return self._wb_ds.__rept_html__()

    def __len__(self):
        return len(self._time_list)

    def __getitem__(self, idx):
        predictors_list = []
        for (var_name, var_pl), ds1 in self._ds_dict.items():
            # get the time for this index
            selected_time = ds1.time[idx].values
            var_stats = self._stats_dict[(var_name, var_pl)]
            normal_da = (ds1[var_name].loc[{'time': selected_time}] - var_stats['mean']) / var_stats['std']
            predictors_list += [normal_da.to_numpy()]          
        
        # use numpy.stack to compose a numpy array with dimension height x width x channels , which is latitude x longitude x var
        select_array = numpy.stack (
            predictors_list,
            axis=-1, # so that the channels i.e. variables dimension is the last dimension
        )
        select_tensor = torch.tensor(
            select_array,
            dtype=torch.float32,
        )
        return select_tensor



In [151]:
wb_ds1 = WeatherbenchDataset(weatherbench_dir, 
                       (datetime.datetime(1980,1,1,0,0), datetime.datetime(1980,1,5,0,0)), 
                       {'temperature': [850, 500], 'geopotential': [500]}, 
                       resolution_dict[5.625],
                       is_train=True,
                      )

temperature_850
temperature_500
geopotential_500


In [152]:
wb_ds1

In [153]:
len(wb_ds1)

97

In [155]:
wb_ds1[3]

tensor([[[-1.0813, -1.2951, -1.4087],
         [-1.0627, -1.2997, -1.4087],
         [-1.0625, -1.2994, -1.4077],
         ...,
         [-1.0874, -1.2379, -1.3962],
         [-1.0950, -1.2637, -1.4035],
         [-1.0893, -1.2832, -1.4066]],

        [[-1.0516, -1.1419, -1.3826],
         [-0.9832, -1.1219, -1.3909],
         [-0.9172, -1.0947, -1.3909],
         ...,
         [-1.0621, -1.0229, -1.3617],
         [-1.0010, -1.0750, -1.3659],
         [-1.0022, -1.1257, -1.3732]],

        [[-0.8930, -0.9975, -1.3397],
         [-0.9254, -1.0316, -1.3387],
         [-0.8973, -1.0105, -1.3241],
         ...,
         [-0.8299, -0.9187, -1.2645],
         [-0.8924, -0.9238, -1.3021],
         [-1.0353, -0.9667, -1.3272]],

        ...,

        [[-1.9194, -1.7776, -1.5801],
         [-1.6273, -1.7577, -1.6731],
         [-1.5650, -1.7248, -1.7264],
         ...,
         [-1.5914, -1.4493, -1.2405],
         [-1.6389, -1.5665, -1.3565],
         [-1.7066, -1.7202, -1.4714]],

        [[

In [154]:
wb_ds1[3].shape

torch.Size([32, 64, 3])

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self,
                 input_height = 501,
                 input_width = 601,
                 kernel_size = 4,
                 stride=2,
                 input_channel_count = 2,
                 output_channel_count = 2,
                 latent_dim=300):
        super(AutoEncoder, self).__init__()

        self.input_width = input_width
        self.input_height = input_height
        self.input_channel_count = input_channel_count
        self.output_channel_count = output_channel_count

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=self.input_channel_count, out_channels=16, kernel_size=kernel_size, stride = stride, padding = 1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride =2, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=7),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=7),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=16, out_channels=self.output_channel_count, kernel_size=kernel_size, stride=stride, padding=1, output_padding=1),
            nn.Sigmoid()
        )


    def forward(self, x):

        # Get latent representation
        latent = self.encoder(x)

        # Reconstruct input
        reconstructed = self.decoder(latent)

        return reconstructed

## Exercises
for students to try that do not have solutions but maybe have an answer or benchmark to facilitate understanding


### Next steps or potential follow on material



###  Exmaples of Use


### Data statement
###     References
